# Retail Revenue Forecasting — Daily Coffee Sales

**End-to-end time-series forecasting project | PostgreSQL · Pandas · Scikit-learn · XGBoost · Skforecast**

### Business objective
Predict **daily revenue** from coffee sales to support short-term financial planning and business decisions.

- **Dataset:** synthetic daily sales of one 500 g coffee product, January 2024–December 2025 (731 observations).
- **Target:** `revenue = unit_price × units_sold` (BRL per day).
- **One-day-ahead models:** seasonal-naive baseline, Random Forest, Gradient Boosting, and XGBoost.
- **Model selection:** chronological holdout plus time-series cross-validation for XGBoost tuning.
- **Future forecasting:** configurable **N-day recursive forecast** with Skforecast and explicit assumptions for future prices, promotions, and holidays.

> **Important:** One-step-ahead test MAE is **not** the measured error of an N-day recursive forecast. The latter requires separate multi-step backtesting. All source data are synthetic and intended for learning/portfolio demonstration.

## 1. Setup and data source

The notebook runs from the included CSV by default. To load the same table from PostgreSQL, set `DATA_SOURCE=postgres` in a local `.env` file and provide connection variables. Never commit `.env` to a public repository.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from xgboost import XGBRegressor

load_dotenv()
pd.set_option("display.max_columns", None)

DATA_SOURCE = os.getenv("DATA_SOURCE", "csv").lower()  # "csv" or "postgres"
CSV_PATH = Path("demanda_cafe_diaria_2024_2025.csv")
RANDOM_STATE = 42

In [ ]:
# Original Portuguese data fields are mapped to English for readability.
COLUMN_MAP = {
    "data": "date",
    "produto": "product",
    "preco_unitario": "unit_price",
    "promocao": "promotion",
    "desconto_percentual": "discount_pct",
    "feriado_indicador": "holiday_flag",
    "unidades_vendidas": "units_sold",
}

if DATA_SOURCE == "postgres":
    required_vars = ["DB_USER", "DB_PASSWORD", "DB_HOST", "DB_NAME"]
    missing_vars = [key for key in required_vars if not os.getenv(key)]
    if missing_vars:
        raise ValueError(f"Missing database settings in .env: {missing_vars}")

    db_url = URL.create(
        drivername="postgresql+psycopg",
        username=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
        host=os.getenv("DB_HOST"),
        port=int(os.getenv("DB_PORT", "5432")),
        database=os.getenv("DB_NAME"),
    )
    engine = create_engine(db_url)
    raw_df = pd.read_sql_query("SELECT * FROM demanda_dia ORDER BY data", con=engine)
elif DATA_SOURCE == "csv":
    raw_df = pd.read_csv(CSV_PATH)
else:
    raise ValueError("DATA_SOURCE must be 'csv' or 'postgres'.")

df = raw_df.rename(columns=COLUMN_MAP)
df["date"] = pd.to_datetime(df["date"])
df["unit_price"] = df["unit_price"].astype(float)
df = df.sort_values("date").reset_index(drop=True)

print(f"Source: {DATA_SOURCE} | Observations: {len(df)}")
display(df.head())

## 2. Data quality and exploratory data analysis

Before modeling, check the time index, duplicates, missing values, and the target distribution. This project contains one product and should have exactly one observation per calendar day.

In [ ]:
required_columns = list(COLUMN_MAP.values())
missing_columns = sorted(set(required_columns) - set(df.columns))
assert not missing_columns, f"Missing required columns: {missing_columns}"
assert df["product"].nunique() == 1, "This notebook models one product only."
assert not df.duplicated("date").any(), "Duplicate daily records detected."

expected_dates = pd.date_range(df["date"].min(), df["date"].max(), freq="D")
missing_dates = expected_dates.difference(pd.DatetimeIndex(df["date"]))
assert missing_dates.empty, f"Missing calendar dates: {missing_dates[:5].tolist()}"

print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print("Missing values by column:")
display(df.isna().sum().to_frame("missing_count"))

In [ ]:
# Revenue is the business outcome being forecast.
df["revenue"] = df["unit_price"] * df["units_sold"]
df["day_of_week"] = df["date"].dt.dayofweek  # Monday=0, Sunday=6
df["month"] = df["date"].dt.month

summary = df[["unit_price", "units_sold", "revenue"]].describe().round(2)
display(summary)

In [ ]:
# Revenue evolution: the unshifted rolling mean is used ONLY for visualization.
plot_rolling_mean_7 = df["revenue"].rolling(window=7).mean()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df["date"], df["revenue"], alpha=0.5, label="Daily revenue")
ax.plot(df["date"], plot_rolling_mean_7, linewidth=2, label="7-day rolling average")
ax.set(title="Daily Coffee Revenue and 7-Day Rolling Average", xlabel="Date", ylabel="Revenue (BRL)")
ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
# Demand distribution and weekly seasonality.
weekday_labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
weekday_revenue = df.groupby("day_of_week")["revenue"].mean().reindex(range(7))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(df["revenue"], bins=25, edgecolor="white")
axes[0].set(title="Daily Revenue Distribution", xlabel="Revenue (BRL)", ylabel="Days")
axes[1].bar(weekday_labels, weekday_revenue)
axes[1].set(title="Average Revenue by Weekday", xlabel="Weekday", ylabel="Average revenue (BRL)")
plt.tight_layout()
plt.show()

In [ ]:
# Compare promotions and non-promotions (association, not a causal estimate).
promotion_summary = (
    df.groupby("promotion")["revenue"]
    .agg(days="count", average_daily_revenue="mean", total_revenue="sum")
    .round(2)
)
display(promotion_summary)

## 3. Feature engineering

The model predicts revenue for day **t** using information available **before** day t, plus same-day features that are assumed to be known in advance (calendar, planned price, and promotions).

- `revenue_lag_1`: observed revenue on day t−1
- `revenue_lag_7`: observed revenue on day t−7
- `revenue_rolling_mean_7`: mean observed revenue over days t−7 through t−1

The `shift(1)` prevents the current day's target from leaking into the rolling average. `units_sold` is excluded because it would reveal the target together with the price.

In [ ]:
df["revenue_lag_1"] = df["revenue"].shift(1)
df["revenue_lag_7"] = df["revenue"].shift(7)
df["revenue_rolling_mean_7"] = df["revenue"].shift(1).rolling(window=7).mean()

feature_columns = [
    "day_of_week",
    "month",
    "revenue_lag_1",
    "revenue_lag_7",
    "revenue_rolling_mean_7",
    "unit_price",
    "promotion",
    "holiday_flag",
]

data_model = df.dropna(subset=feature_columns + ["revenue"]).copy()
X = data_model[feature_columns]
y = data_model["revenue"]

print(f"Usable observations: {len(data_model)} / {len(df)}")
display(data_model[["date", "revenue"] + feature_columns].head(5))

## 4. Chronological train/test split

The oldest **80%** of usable observations are used for training; the most recent **20%** form the test period. There is no random shuffling.

**Evaluation assumption:** each test-date forecast uses actual historical revenue through the previous day, as if the model were run daily. Actual prices and promotions from the test period are treated as *known schedules*; in deployment, they must be supplied prospectively.

In [ ]:
split_index = int(len(data_model) * 0.80)
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]
train_dates = data_model["date"].iloc[:split_index]
test_dates = data_model["date"].iloc[split_index:]

print(f"Training: {len(X_train)} days ({train_dates.min().date()} to {train_dates.max().date()})")
print(f"Test:     {len(X_test)} days ({test_dates.min().date()} to {test_dates.max().date()})")

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(train_dates, y_train, label="Train")
ax.plot(test_dates, y_test, label="Test")
ax.set(title="Chronological Train/Test Split", xlabel="Date", ylabel="Revenue (BRL)")
ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 5. Baseline and machine learning models

A **seasonal naive** baseline predicts each day's revenue using the revenue on the same weekday one week earlier. We then compare three tree-based regressors on the **same chronological test period**.

In [ ]:
# Seasonal naive benchmark: predicted revenue(t) = actual revenue(t - 7).
baseline_predictions = X_test["revenue_lag_7"].to_numpy()

models = {
    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        max_depth=10,
        min_samples_leaf=3,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.02,
        max_depth=3,
        random_state=RANDOM_STATE,
    ),
    "XGBoost": XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}

predictions = {"Seasonal Naive": baseline_predictions}
for model_name, estimator in models.items():
    estimator.fit(X_train, y_train)
    predictions[model_name] = estimator.predict(X_test)

In [ ]:
# Use the same MAE, RMSE, and R² definitions for every model.
metrics = []
for name, y_hat in predictions.items():
    metrics.append({
        "model": name,
        "MAE (BRL)": mean_absolute_error(y_test, y_hat),
        "RMSE (BRL)": np.sqrt(mean_squared_error(y_test, y_hat)),
        "R2": r2_score(y_test, y_hat),
    })

metrics_df = pd.DataFrame(metrics).sort_values("MAE (BRL)").reset_index(drop=True)
display(metrics_df.round(3))

In [ ]:
# Visual comparison on the held-out period.
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(test_dates, y_test, label="Actual revenue", color="black", linewidth=1.5)
ax.plot(test_dates, predictions["Random Forest"], label="Random Forest", alpha=0.75)
ax.plot(test_dates, predictions["Gradient Boosting"], label="Gradient Boosting", alpha=0.75)
ax.plot(test_dates, predictions["XGBoost"], label="XGBoost", alpha=0.75)
ax.set(title="One-Day-Ahead Predictions on the Test Period", xlabel="Date", ylabel="Revenue (BRL)")
ax.legend(ncol=2)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 6. XGBoost hyperparameter tuning

`RandomizedSearchCV` searches candidate XGBoost settings using **expanding-window time-series cross-validation** on the training portion only. Hyperparameters are selected using the average validation MAE; the test period is not used inside the search.

The search reproduces the original experiment: 50 candidate combinations evaluated across four folds (200 fits). The reported holdout model comparisons are still exploratory because multiple models are inspected on the same test period.

In [ ]:
parameter_distributions = {
    "n_estimators": [50, 100, 200, 300, 400, 500],
    "max_depth": [2, 3, 4, 5, 6, 7, 8],
    "learning_rate": [0.01, 0.02, 0.03, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1],
    "subsample": [0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0],
    "colsample_bytree": [0.7, 0.85, 1.0],
    "min_child_weight": [1, 3, 5],
}

cv = TimeSeriesSplit(n_splits=4, test_size=60)
search = RandomizedSearchCV(
    estimator=XGBRegressor(
        objective="reg:squarederror",
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=1,
    ),
    param_distributions=parameter_distributions,
    n_iter=50,
    scoring="neg_mean_absolute_error",
    cv=cv,
    random_state=32,
    n_jobs=-1,
    verbose=1,
    refit=True,
)
search.fit(X_train, y_train)

print("Best hyperparameters:", search.best_params_)
print(f"Cross-validation MAE: BRL {-search.best_score_:.2f}")

In [ ]:
# Evaluate the tuned model on the same held-out dates.
tuned_xgb = search.best_estimator_
tuned_predictions = tuned_xgb.predict(X_test)

metrics_df = pd.concat([
    metrics_df,
    pd.DataFrame([{
        "model": "XGBoost (Tuned)",
        "MAE (BRL)": mean_absolute_error(y_test, tuned_predictions),
        "RMSE (BRL)": np.sqrt(mean_squared_error(y_test, tuned_predictions)),
        "R2": r2_score(y_test, tuned_predictions),
    }]),
], ignore_index=True).sort_values("MAE (BRL)").reset_index(drop=True)

display(metrics_df.round(3))

In [ ]:
# Built-in importance reflects predictive split contributions, not causality.
feature_importance = (
    pd.Series(tuned_xgb.feature_importances_, index=feature_columns)
    .sort_values(ascending=True)
)

fig, ax = plt.subplots(figsize=(9, 4))
feature_importance.plot.barh(ax=ax)
ax.set(title="Tuned XGBoost — Feature Importance", xlabel="Relative importance", ylabel="Feature")
plt.tight_layout()
plt.show()

## 7. Configurable multi-day forecasting with Skforecast

To forecast **N future days from a single forecast origin**, Skforecast recursively feeds earlier predictions into the lag/rolling features. The forecaster is re-trained on **all available historical observations** after model evaluation, using the tuned XGBoost configuration.

Future calendar variables are known; future **price**, **promotion**, and **holiday** information must be provided as assumptions or an actual business calendar. This demonstration assumes a constant regular price, no promotion, and no holidays. Changing these assumptions changes the forecast.

> **Forecast origin:** the last date in the dataset (2025-12-31 in the supplied sample). The forecasts do not start from your computer's current date unless you update the underlying data.

In [ ]:
from skforecast.preprocessing import RollingFeatures
from skforecast.recursive import ForecasterRecursive

# User-controlled forecast horizon.
FORECAST_DAYS = 10  # Change to 7, 30, 60, etc.
assert isinstance(FORECAST_DAYS, int) and FORECAST_DAYS > 0

# The target and exogenous variables must share the same daily time index.
daily = df.set_index("date").asfreq("D")
historical_revenue = daily["revenue"].astype(float)
exogenous_columns = ["day_of_week", "month", "unit_price", "promotion", "holiday_flag"]
historical_exog = daily[exogenous_columns].copy()

forecaster = ForecasterRecursive(
    estimator=XGBRegressor(
        **search.best_params_,
        objective="reg:squarederror",
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    lags=[1, 7],
    window_features=RollingFeatures(stats=["mean"], window_sizes=[7]),
)
forecaster.fit(y=historical_revenue, exog=historical_exog)

In [ ]:
# Explicit future business scenario (replace with your actual price/promo/holiday plan).
future_dates = pd.date_range(
    start=historical_revenue.index.max() + pd.Timedelta(days=1),
    periods=FORECAST_DAYS,
    freq="D",
)
regular_prices = daily.loc[daily["promotion"] == 0, "unit_price"]
assumed_unit_price = float(
    regular_prices.iloc[-1] if not regular_prices.empty else daily["unit_price"].iloc[-1]
)

future_exog = pd.DataFrame({
    "day_of_week": future_dates.dayofweek,
    "month": future_dates.month,
    "unit_price": assumed_unit_price,
    "promotion": 0,
    "holiday_flag": 0,
}, index=future_dates)[exogenous_columns]

future_predictions = forecaster.predict(steps=FORECAST_DAYS, exog=future_exog)
forecast_df = future_predictions.rename("predicted_revenue").rename_axis("date").reset_index()
display(forecast_df)

In [ ]:
# Plot recent observations followed by the recursive multi-step forecast.
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(historical_revenue.tail(60).index, historical_revenue.tail(60), label="Observed revenue")
ax.plot(future_predictions.index, future_predictions.values, "--", label="Recursive forecast")
ax.axvline(historical_revenue.index.max(), linestyle=":", color="gray", label="Forecast origin")
ax.set(
    title=f"Revenue Forecast: Next {FORECAST_DAYS} Days",
    xlabel="Date",
    ylabel="Revenue (BRL)",
)
ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 8. Conclusions and limitations

- **One-day-ahead comparison:** use the model comparison table above to judge relative MAE, RMSE, and R² on the same holdout period.
- **Recursive multi-day forecasts:** the forecast length is controlled by `FORECAST_DAYS`, but its error is **not** established by the one-day test metrics.
- **Forecast assumptions:** planned prices, promotions, and holidays are required for realistic future predictions; the current scenario is intentionally simplified.
- **Selection bias:** because several models are compared on one holdout, a fresh untouched holdout or proper rolling-origin backtest is recommended before claiming production-level accuracy.
- **Dataset scope:** one synthetic product; no causal conclusions and no guarantee of performance on real retail data.

**Suggested next improvement:** perform rolling-origin backtesting at 7-, 14-, and 30-day horizons, comparing the recursive forecast with an equally defined seasonal baseline.